In [1]:
from Data_checker import DataChecker
import pandas as pd
import numpy as np
import pickle
from Feature_selector import FeatureSelector
from L1_model_zoo import *
import os
os.environ["LOKY_MAX_CPU_COUNT"] = "12"  # 限制最多使用 4 個核心
from Search_hyper_params import Serch_hyperParams
from train_models import Training_models


c:\Users\E4-159\anaconda3\envs\torch_\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
all_train_data = pd.read_csv(rf'C:\Users\E4-159\Documents\py_surr\imbd2025\初\final_dataset_with_statics.csv')
# drop Disp. X and Disp. Z columns for y
X = all_train_data.drop(columns=['Time','Disp. X', 'Disp. Z', '日期'])
y_x = all_train_data['Disp. X']
y_z = all_train_data['Disp. Z']
'''pick Categorical features: ['轉速 (rpm)', '轉速 (rpm).1', '溫度']
Numerical features: ['Spindle Motor', 'X Motor', 'Z Motor', 'PT01_mean', 'PT01_skew', 'PT02_std', 'PT03_std', 'PT03_skew', 'PT06_std', 'PT07_skew', 'PT07_kurtosis', 'PT09_skew', 'PT11_skew', 'TC02_std', 'TC02_skew', 'TC03_skew', 'TC04_skew', 'TC06_kurtosis', 'TC07_median', 'TC07_quantile_75%', 'TC07_kurtosis', 'X Motor_skew', 'X Motor_kurtosis', 'Z Motor_skew']'''
X = X[['轉速 (rpm)', '轉速 (rpm).1', '溫度','Spindle Motor', 'X Motor', 'Z Motor', 'PT01_mean', 'PT01_skew', 'PT02_std', 'PT03_std', 'PT03_skew', 'PT06_std', 'PT07_skew', 'PT07_kurtosis', 'PT09_skew', 'PT11_skew', 'TC02_std', 'TC02_skew', 'TC03_skew', 'TC04_skew', 'TC06_kurtosis', 'TC07_median', 'TC07_quantile_75%', 'TC07_kurtosis', 'X Motor_skew', 'X Motor_kurtosis', 'Z Motor_skew']]
# x size
print("X shape:", X.shape)
print("Number of features:", X.shape[1])
    
# Create typeofFeatures to match the number of features
# Assuming first 25 are numerical and remaining are categorical
#typeofFeatures_new = [1] * (X.shape[1] - 11) + [0] * 11
typeofFeatures_new = [0] * 3 + [1] * (X.shape[1] - 3)  # First 3 are categorical, rest are numerical
  
# Create a test dataset with the same structure as X (DataFrame) instead of numpy array
X_test_dummy = pd.DataFrame(np.zeros((X.shape[0], X.shape[1])), columns=X.columns)
    
data_checker = DataChecker(X=X, y=y_z, mode='reg', typeofFeatures=typeofFeatures_new, X_test=X_test_dummy)
data_checker.varify_data_types()
data_checker.apply_transformations(use_target_encoder=False)  # Use False to skip Target Encoding for now
kfold_splits = data_checker.get_folds(n_splits=5, n_repeats=10)    

do = Serch_hyperParams(train=kfold_splits['train_splits'], val=kfold_splits['valid_splits'], mode='reg', use_gpu=True)
xgboost_params = do.search_in_xgboost(n=20)   #done
randomforest_params = do.search_in_randomforest(n=20)
catboost_params = do.search_in_catboost(n=20)
extratrees_params = do.search_in_extratrees(n=20)
lgbm_params = do.search_in_lgbm(n=20)
# into 1 dict
'''all_params = {
    'xgboost': xgboost_params,
    'randomforest': randomforest_params,
    'catboost': catboost_params,
    'extratrees': extratrees_params,
    'lgbm': lgbm_params
}'''
# train models
trainer = Training_models(train=kfold_splits['train_splits'], val=kfold_splits['valid_splits'], params=None)
trainer.train_models()
all_trained_models = trainer.trained_models

X shape: (25495, 27)
Number of features: 27
Categorical features: ['轉速 (rpm)', '轉速 (rpm).1', '溫度']
Numerical features: ['Spindle Motor', 'X Motor', 'Z Motor', 'PT01_mean', 'PT01_skew', 'PT02_std', 'PT03_std', 'PT03_skew', 'PT06_std', 'PT07_skew', 'PT07_kurtosis', 'PT09_skew', 'PT11_skew', 'TC02_std', 'TC02_skew', 'TC03_skew', 'TC04_skew', 'TC06_kurtosis', 'TC07_median', 'TC07_quantile_75%', 'TC07_kurtosis', 'X Motor_skew', 'X Motor_kurtosis', 'Z Motor_skew']
No numerical features found, skipping numerical transformations.
No categorical features found, skipping target encoding.
Repeated KFold splits with holdout created successfully!
Number of CV splits per target: 50
Training set size for first fold: 18356
Validation set size for first fold: 4589
Holdout set size: 2550
Total original data size: 25495


[I 2025-08-16 02:12:00,233] A new study created in memory with name: no-name-71a32523-c84e-4897-874b-98cd2eaf4f10
c:\Users\E4-159\anaconda3\envs\torch_\Lib\site-packages\xgboost\core.py:729: UserWarning: [02:12:02] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:58: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)
[I 2025-08-16 02:13:08,937] Trial 0 finished with value: 1.8448663981697637 and parameters: {'n_estimators': 800, 'learning_rate': 0.13, 'max_depth': 4, 'subsample': 0.6, 'colsample_bytree': 0.55, 'reg_alpha': 0.8, 'reg_lambda': 0.9}. Best is trial 0 with value: 1.8448663981697637.
[I 2025-08-16

Training xgboost...
xgboost trained with RMSE: 1.6832527675985662
Training randomforest...
randomforest trained with RMSE: 1.6423696162582382
Training catboost...
catboost trained with RMSE: 1.7076123207578147
Training extratrees...
extratrees trained with RMSE: 1.6277174850243923
Training lgbm...
lgbm trained with RMSE: 1.9222407391078478


In [3]:
print("Trained models:", all_trained_models.keys())

Trained models: dict_keys(['xgboost', 'randomforest', 'catboost', 'extratrees', 'lgbm'])


In [4]:
print(len(all_trained_models['xgboost']))

50


In [5]:
print(len(all_trained_models['xgboost']))

50


In [6]:
print(len(kfold_splits['train_splits'][0]))

50


In [7]:
# concat models to kfold_splits by adding all_trained_models dict 
kfold_splits['trained_models'] = all_trained_models

In [8]:
print(kfold_splits['train_splits'][0][1].keys())  # Print the first fold's first model's keys

dict_keys(['X', 'y', 'scaler', 'gaussian', 'encoder'])


In [9]:
print(kfold_splits['holdout_split'][0]['X'].shape)  # Print the keys of the holdout split

(2550, 27)


In [10]:
from pprint import pprint
pprint(kfold_splits)


{'holdout_split': [{'X':        轉速 (rpm)  轉速 (rpm).1  溫度  Spindle Motor  X Motor  Z Motor  PT01_mean  \
20325         0           0   1           24.0     24.0     25.0     18.415   
20509         3           1  15           37.0     29.0     29.0     27.633   
6137          1           1  14           30.0     25.0     26.0     24.097   
1246          0           3  12           37.0     29.0     29.0     22.526   
21657         0           1   6           24.0     24.0     24.0     20.839   
...         ...         ...  ..            ...      ...      ...        ...   
10392         3           0  18           39.0     30.0     32.0     27.273   
2762          0           0   6           27.0     25.0     26.0     21.267   
20690         3           1  15           36.0     31.0     31.0     27.633   
12421         3           0   9           37.0     29.0     29.0     21.239   
7568          0           3  16           36.0     32.0     32.0     27.893   

       PT01_skew  PT02_std

In [11]:
pprint(all_trained_models)

{'catboost': [<catboost.core.CatBoostRegressor object at 0x000001B4FB9AAC30>,
 'extratrees': [ExtraTreesRegressor(),
                ExtraTreesRegressor(),
                ExtraTreesRegressor(),
                ExtraTreesRegressor(),
                ExtraTreesRegressor(),
                ExtraTreesRegressor(),
                ExtraTreesRegressor(),
                ExtraTreesRegressor(),
                ExtraTreesRegressor(),
                ExtraTreesRegressor(),
                ExtraTreesRegressor(),
                ExtraTreesRegressor(),
                ExtraTreesRegressor(),
                ExtraTreesRegressor(),
                ExtraTreesRegressor(),
                ExtraTreesRegressor(),
                ExtraTreesRegressor(),
                ExtraTreesRegressor(),
                ExtraTreesRegressor(),
                ExtraTreesRegressor(),
                ExtraTreesRegressor(),
                ExtraTreesRegressor(),
                ExtraTreesRegressor(),
                ExtraTree

In [12]:
from Kfold_predictor import KFoldPredictor
kfold_predictor = KFoldPredictor(kfold_splits=kfold_splits, models=all_trained_models, types_of_features=typeofFeatures_new)
kfold_predictor.fit_holdout()

c:\Users\E4-159\Documents\GitHub\IMBD_helper\Kfold_predictor.py:61: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[  0.62358997 -15.57175847   0.31001432 ... -15.57175847 -15.57175847
   0.62358997]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  holdout_X.iloc[:, categorical_indices] = encoder.transform(holdout_X.iloc[:, categorical_indices])
c:\Users\E4-159\Documents\GitHub\IMBD_helper\Kfold_predictor.py:61: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[-4.17606869 -7.99836523 -7.99836523 ... -7.99836523 -4.17606869
 -8.96077627]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  holdout_X.iloc[:, categorical_indices] = encoder.transform(holdout_X.iloc[:, categorical_indices])
c:\Users\E4-159\Documents\GitHub\IMBD_helper\Kfold_predictor.py:61: FutureWarning: Set

RMSE for xgboost: 3.4620362341871727
RMSE for randomforest: 1.6629634455814868
RMSE for catboost: 1.739972513242279
RMSE for extratrees: 1.7578954527554371
RMSE for lgbm: 1.9692081183532497
Weights calculated: [0.11364849 0.23659881 0.22612724 0.22382171 0.19980375]
Final RMSE: 1.7595126415540439
